In [1]:
pip install torch torchvision

Note: you may need to restart the kernel to use updated packages.


In [2]:
local_root = "./miniddsm2"

In [3]:
!mc ls s3/dimitri/stat_app/
!mc ls s3/dimitri/stat_app/miniddsm2


]11;?\mc: Configuration written to `/home/onyxia/.mc/config.json`. Please update your access credentials.
mc: Successfully created `/home/onyxia/.mc/share`.
mc: Initialized share uploads `/home/onyxia/.mc/share/uploads.json` file.
mc: Initialized share downloads `/home/onyxia/.mc/share/downloads.json` file.
[2026-01-27 14:08:49 UTC]    39B STANDARD .keep
[2026-02-20 14:58:55 UTC]     0B CMMD2022/
[2026-02-20 14:58:55 UTC]     0B VinDr/
[2026-02-20 14:58:55 UTC]     0B miniddsm2/

== WARN: `minio.lab.sspcloud.fr` certificate will expire in 2026-02-26 12:48:48 +0000 UTC. Renew soon to avoid outage.

]11;?\[2026-02-20 14:59:00 UTC]     0B Data-MoreThanTwoMasks/
[2026-02-20 14:59:00 UTC]     0B MINI-DDSM-Complete-JPEG-8/
[2026-02-20 14:59:00 UTC]     0B MINI-DDSM-Complete-PNG-16/

== WARN: `minio.lab.sspcloud.fr` certificate will expire in 2026-02-26 12:48:48 +0000 UTC. Renew soon to avoid outage.



In [4]:
!mc mirror \
"s3/dimitri/stat_app/miniddsm2/MINI-DDSM-Complete-JPEG-8" \
"./miniddsm2/MINI-DDSM-Complete-JPEG-8"


...OMB.16_PGM: 3.96 GiB / 3.96 GiB ┃▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓┃ 115.77 MiB/s 34s
== WARN: `minio.lab.sspcloud.fr` certificate will expire in 2026-02-26 12:48:48 +0000 UTC. Renew soon to avoid outage.



In [5]:
!ls -la ./miniddsm2
!ls -la ./miniddsm2/MINI-DDSM-Complete-JPEG-8 | head


total 12
drwxrwsr-x 3 onyxia users 4096 Feb  4 18:37 .
drwxrwsr-x 5 onyxia users 4096 Feb  4 23:10 ..
drwxrwsr-x 5 onyxia users 4096 Feb  4 18:37 MINI-DDSM-Complete-JPEG-8
total 1576
drwxrwsr-x   5 onyxia users    4096 Feb  4 18:37 .
drwxrwsr-x   3 onyxia users    4096 Feb  4 18:37 ..
drwxrwsr-x 673 onyxia users   12288 Feb  4 18:37 Benign
-rw-rw-r--   1 onyxia users 1150890 Feb  4 18:37 BoundaryMask.png
drwxrwsr-x 681 onyxia users   16384 Feb  4 18:37 Cancer
-rw-rw-r--   1 onyxia users  411637 Feb  4 18:37 DataWMask.xlsx
drwxrwsr-x 604 onyxia users   12288 Feb  4 18:38 Normal


In [5]:
import os
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.datasets import ImageFolder
from torchvision import transforms

data_root = "./miniddsm2/MINI-DDSM-Complete-JPEG-8"

train_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((700, 700)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((700, 700)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

base_ds = ImageFolder(root=data_root, transform=train_tf)
print("class_to_idx:", base_ds.class_to_idx)  # doit être {'Benign':0,'Cancer':1,'Normal':2}

benign_idx = base_ds.class_to_idx["Benign"]
cancer_idx = base_ds.class_to_idx["Cancer"]
normal_idx = base_ds.class_to_idx["Normal"]

class BinaryDataset(Dataset):
    def __init__(self, base, pos_idxs):
        self.base = base
        self.pos = set(pos_idxs)
        self.classes = ["Negative", "Positive"]
        self.class_to_idx = {"Negative": 0, "Positive": 1}

    def __len__(self):
        return len(self.base)

    def __getitem__(self, i):
        x, y = self.base[i]
        y_bin = 1 if y in self.pos else 0
        return x, y_bin

full_ds = BinaryDataset(base_ds, pos_idxs=[benign_idx, cancer_idx])
print("Total images:", len(full_ds))


class_to_idx: {'Benign': 0, 'Cancer': 1, 'Normal': 2}
Total images: 10873


In [15]:
from torchvision.datasets import ImageFolder
from torch.utils.data import Dataset

base_ds = ImageFolder(root=data_root, transform=train_tf)

class BinaryDataset(Dataset):
    def __init__(self, base_dataset):
        self.base = base_dataset

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        x, y = self.base[idx]
        y_bin = 1 if y in (0, 1) else 0
        return x, y_bin

binary_ds = BinaryDataset(base_ds)


vérifier la relabelisation

In [16]:
import random
from collections import Counter

idxs = random.sample(range(len(binary_ds)), 200)
labels = [binary_ds[i][1] for i in idxs]
Counter(labels)


Counter({1: 153, 0: 47})

In [17]:
from torch.utils.data import random_split, DataLoader

total = len(full_ds)
train_size = int(0.7 * total)
val_size   = int(0.15 * total)
test_size  = total - train_size - val_size

g = torch.Generator().manual_seed(42)
train_ds, val_ds, test_ds = random_split(full_ds, [train_size, val_size, test_size], generator=g)

# pas d'augmentations en val/test
val_ds.dataset.base.transform = val_tf
test_ds.dataset.base.transform = val_tf

bs = 8
train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)

print(train_size, val_size, test_size)


7611 1630 1632


In [18]:
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

model = resnet18(weights=ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)


device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/onyxia/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 97.2MB/s]


In [19]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


In [20]:
from tqdm import tqdm

def train_one_epoch():
    model.train()
    loss_sum, correct, n = 0.0, 0, 0
    for x, y in tqdm(train_loader, desc="Train"):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        n += x.size(0)
    return loss_sum / n, correct / n

@torch.no_grad()
def eval_epoch(loader, desc):
    model.eval()
    loss_sum, correct, n = 0.0, 0, 0
    for x, y in tqdm(loader, desc=desc):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss_sum += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        n += x.size(0)
    return loss_sum / n, correct / n

best_val = -1.0
save_path = "best_resnet18_miniddsm2_three_classes.pth"
epochs = 10

for epoch in range(1, epochs + 1):
    print(f"\nEpoch {epoch}/{epochs}")
    tr_loss, tr_acc = train_one_epoch()
    va_loss, va_acc = eval_epoch(val_loader, "Val")
    print(f"train loss={tr_loss:.4f} acc={tr_acc:.4f} | val loss={va_loss:.4f} acc={va_acc:.4f}")

    if va_acc > best_val:
        best_val = va_acc
        torch.save(model.state_dict(), save_path)
        print(f"✅ saved best (val acc={best_val:.4f})")



Epoch 1/10


Val: 100%|██████████| 204/204 [00:57<00:00,  3.54it/s]


train loss=0.2131 acc=0.9116 | val loss=0.1608 acc=0.9466
✅ saved best (val acc=0.9466)

Epoch 2/10


Val: 100%|██████████| 204/204 [00:55<00:00,  3.70it/s]


train loss=0.1547 acc=0.9365 | val loss=0.2823 acc=0.8755

Epoch 3/10


...

train loss=0.0641 acc=0.9733 | val loss=0.1940 acc=0.9429

Epoch 9/10


Val: 100%|██████████| 204/204 [00:55<00:00,  3.66it/s]


train loss=0.0506 acc=0.9806 | val loss=0.1337 acc=0.9491

Epoch 10/10


Val: 100%|██████████| 204/204 [00:56<00:00,  3.59it/s]

train loss=0.0384 acc=0.9852 | val loss=0.1168 acc=0.9564


In [21]:
from sklearn.metrics import f1_score, confusion_matrix, classification_report

@torch.no_grad()
def test_f1():
    model.load_state_dict(torch.load(save_path, map_location=device))
    model.eval()

    y_true, y_pred = [], []
    for x, y in tqdm(test_loader, desc="Test"):
        x = x.to(device, non_blocking=True)
        logits = model(x)
        preds = logits.argmax(1).cpu().numpy()

        y_true.extend(y.numpy())
        y_pred.extend(preds)

    f1 = f1_score(y_true, y_pred, average="binary", pos_label=1)
    print("F1 (Positive=Benign+Cancer):", f1)
    print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred, target_names=["Negative(Normal)", "Positive(B+C)"]))

test_f1()


Test: 100%|██████████| 204/204 [00:58<00:00,  3.50it/s]

F1 (Positive=Benign+Cancer): 0.9638834951456311
Confusion matrix:
 [[ 298   74]
 [  19 1241]]
                  precision    recall  f1-score   support

Negative(Normal)       0.94      0.80      0.87       372
   Positive(B+C)       0.94      0.98      0.96      1260

        accuracy                           0.94      1632
       macro avg       0.94      0.89      0.91      1632
    weighted avg       0.94      0.94      0.94      1632

